# OpenCV 学习笔记

OpenCV（Open Source Computer Vision Library）是一个开源的计算机视觉和机器学习软件库，包含超过2500种算法，广泛应用于图像处理、视频分析、物体检测、面部识别等领域。

本笔记系统介绍 OpenCV 的核心概念和常用函数。

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
# 在 Jupyter 中显示图像
def show(img, title="image"):
    plt.figure(figsize=(6, 4))
    if len(img.shape) == 3:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.show()

---

## 1. 图像基础概念

### 1.1 OpenCV 图像读取与显示

OpenCV 中图像以 NumPy 数组形式存储。彩色图像是 BGR 格式（注意不是 RGB），每个像素由蓝、绿、红三个通道组成。

**常用函数：**
- `cv2.imread()` - 读取图像
- `cv2.imwrite()` - 保存图像
- `cv2.imshow()` - 显示图像（主要用于测试）

In [ ]:
# 读取图像
# cv2.IMREAD_COLOR = 1   彩色图像 (BGR)
# cv2.IMREAD_GRAYSCALE = 0 灰度图像
# cv2.IMREAD_UNCHANGED = -1  包含Alpha通道

img = cv2.imread("test.jpg", cv2.IMREAD_COLOR)

# 保存图像
cv2.imwrite("output.jpg", img)

# 获取图像基本信息
print(f"图像形状: {img.shape}")  # (高度, 宽度, 通道数)
print(f"图像数据类型: {img.dtype}")
print(f"图像尺寸: {img.size} 像素")

### 1.2 图像通道与颜色空间

OpenCV 默认使用 BGR 颜色空间，与 RGB 相反。常用颜色空间包括：

- **BGR** - OpenCV 默认格式
- **RGB** - 标准的红绿蓝格式
- **GRAY** - 灰度图，单通道
- **HSV** - 色相、饱和度、明度，适合颜色检测
- **LAB** - 明度、A分量、B分量

In [ ]:
img = cv2.imread("test.jpg")

# BGR 转 RGB
rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# BGR 转 灰度图
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# BGR 转 HSV
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# BGR 转 LAB
lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

# 分离通道
b, g, r = cv2.split(img)

# 合并通道
merged = cv2.merge([b, g, r])

---

## 2. 图像基础操作

### 2.1 图像缩放与插值

图像缩放使用 `cv2.resize()` 函数，支持多种插值方法：

- `cv2.INTER_LINEAR` - 双线性插值（默认，速度和质量的平衡）
- `cv2.INTER_CUBIC` - 双立方插值（质量高，速度慢）
- `cv2.INTER_NEAREST` - 最近邻插值（速度最快，质量差）
- `cv2.INTER_LANCZOS4` - Lanczos插值（最高质量）

**尺寸计算：**
- 缩小图像时用 `cv2.INTER_AREA` 效果最好
- 放大图像时用 `cv2.INTER_CUBIC` 或 `cv2.INTER_LINEAR`

In [ ]:
img = cv2.imread("test.jpg")

# 指定尺寸缩放
resized = cv2.resize(img, (800, 600))

# 按比例缩放 (fx, fy)
resized2 = cv2.resize(img, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_LINEAR)

# 使用 INTER_AREA 进行缩小（适合图像缩小）
small = cv2.resize(img, (400, 300), interpolation=cv2.INTER_AREA)

# 使用 INTER_CUBIC 进行放大（质量更好）
large = cv2.resize(img, (1600, 1200), interpolation=cv2.INTER_CUBIC)

### 2.2 图像平移

使用仿射变换矩阵实现图像平移。

变换矩阵格式：
```
[1, 0, tx]
[0, 1, ty]
```

其中 tx 是水平方向平移量，ty 是垂直方向平移量。

In [ ]:
img = cv2.imread("test.jpg")
rows, cols = img.shape[:2]

# 定义平移矩阵：向右平移100像素，向下平移50像素
M = np.float32([[1, 0, 100], [0, 1, 50]])

# 应用仿射变换
translated = cv2.warpAffine(img, M, (cols, rows))

# 显示结果
# show(translated, "Translated Image")

### 2.3 图像旋转

使用 `cv2.getRotationMatrix2D()` 创建旋转矩阵，然后用 `cv2.warpAffine()` 应用变换。

参数：
- 旋转中心 (center_x, center_y)
- 旋转角度（正数逆时针，负数顺时针）
- 缩放因子

In [ ]:
img = cv2.imread("test.jpg")
rows, cols = img.shape[:2]

# 创建旋转矩阵：绕中心点旋转45度，缩放因子1
M = cv2.getRotationMatrix2D((cols/2, rows/2), 45, 1)

# 应用旋转
rotated = cv2.warpAffine(img, M, (cols, rows))

# 绕中心旋转，且不裁剪边缘
# 先计算旋转后的图像尺寸
cos = np.abs(M[0, 0])
sin = np.abs(M[0, 1])
new_cols = int((rows * sin) + (cols * cos))
new_rows = int((rows * cos) + (cols * sin))
M = cv2.getRotationMatrix2D((cols/2, rows/2), -30, 1)
M[0, 2] += (new_cols - cols) / 2
M[1, 2] += (new_rows - rows) / 2
rotated2 = cv2.warpAffine(img, M, (new_cols, new_rows))

### 2.4 仿射变换

仿射变换是线性变换，包含平移、旋转、缩放、剪切等。变换后平行线保持平行。

使用 `cv2.getAffineTransform()` 从三对对应点计算仿射变换矩阵。

In [ ]:
img = cv2.imread("test.jpg")
rows, cols = img.shape[:2]

# 定义源图像中的三个点
pts1 = np.float32([[50, 50], [200, 50], [50, 200]])

# 定义目标图像中的三个对应点
pts2 = np.float32([[10, 100], [200, 50], [100, 250]])

# 计算仿射变换矩阵
M = cv2.getAffineTransform(pts1, pts2)

# 应用仿射变换
affine = cv2.warpAffine(img, M, (cols, rows))

### 2.5 透视变换

透视变换可以改变图像的视角，用于矫正梯形畸变。

使用 `cv2.getPerspectiveTransform()` 计算变换矩阵，需要四对对应点。

In [ ]:
img = cv2.imread("test.jpg")
rows, cols = img.shape[:2]

# 定义源图像中的四个点
pts1 = np.float32([[56, 65], [368, 52], [28, 387], [389, 390]])

# 定义目标图像中的四个对应点
pts2 = np.float32([[0, 0], [300, 0], [0, 300], [300, 300]])

# 计算透视变换矩阵
M = cv2.getPerspectiveTransform(pts1, pts2)

# 应用透视变换
perspective = cv2.warpPerspective(img, M, (300, 300))

---

## 3. 图像阈值处理

阈值处理是将图像转换为二值图像的基本方法。

### 3.1 简单阈值

给定一个阈值 T，像素值大于 T 设为最大值，小于 T 设为0（或反之）。

阈值类型：
- `cv2.THRESH_BINARY` - 大于阈值为最大值，其余为0
- `cv2.THRESH_BINARY_INV` - 大于阈值为0，其余为最大值
- `cv2.THRESH_TRUNC` - 大于阈值的截断为阈值，其余不变
- `cv2.THRESH_TOZERO` - 大于阈值的保持不变，其余为0
- `cv2.THRESH_TOZERO_INV` - 大于阈值的为0，其余保持不变

In [ ]:
img = cv2.imread("test.jpg", cv2.IMREAD_GRAYSCALE)

# 简单阈值处理
ret, thresh1 = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
ret, thresh2 = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY_INV)
ret, thresh3 = cv2.threshold(img, 127, 255, cv2.THRESH_TRUNC)
ret, thresh4 = cv2.threshold(img, 127, 255, cv2.THRESH_TOZERO)
ret, thresh5 = cv2.threshold(img, 127, 255, cv2.THRESH_TOZERO_INV)

print(f"阈值: {ret}")

### 3.2 自适应阈值

自适应阈值根据像素邻域的局部特征确定阈值，适用于光照不均匀的图像。

- `cv2.ADAPTIVE_THRESH_MEAN_C` - 邻域均值减去常数 C
- `cv2.ADAPTIVE_THRESH_GAUSSIAN_C` - 邻域加权均值减去常数 C

In [ ]:
img = cv2.imread("test.jpg", cv2.IMREAD_GRAYSCALE)

# 自适应阈值 - 使用均值
thresh_mean = cv2.adaptiveThreshold(
    img, 255, cv2.ADAPTIVE_THRESH_MEAN_C, 
    cv2.THRESH_BINARY, 11, 2
)

# 自适应阈值 - 使用高斯加权均值
thresh_gaussian = cv2.adaptiveThreshold(
    img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
    cv2.THRESH_BINARY, 11, 2
)

# 参数说明：
# blockSize = 11  邻域大小（必须是奇数）
# C = 2           从均值或加权均值中减去的常数

---

## 4. 图像模糊与平滑

图像模糊用于降噪和平滑处理。

### 4.1 平均模糊

使用卷积核内所有像素的平均值替代中心像素值。

In [ ]:
img = cv2.imread("test.jpg")

# 使用 5x5 的卷积核进行平均模糊
blur = cv2.blur(img, (5, 5))

# 使用 10x10 的卷积核
blur2 = cv2.blur(img, (10, 10))

### 4.2 高斯模糊

高斯模糊使用高斯函数生成的卷积核，对图像进行加权平均。边缘保留比平均模糊更好。

参数：卷积核大小（必须是奇数）和标准差（0表示自动计算）。

In [ ]:
img = cv2.imread("test.jpg")

# 高斯模糊 - 5x5卷积核，标准差0（自动）
gaussian = cv2.GaussianBlur(img, (5, 5), 0)

# 高斯模糊 - 9x9卷积核，标准差10
gaussian2 = cv2.GaussianBlur(img, (9, 9), 10)

### 4.3 中值模糊

中值模糊用邻域像素的中值替代中心像素值。对盐椒噪声特别有效。

**注意**：卷积核大小必须是奇数。

In [ ]:
img = cv2.imread("test.jpg")

# 中值模糊 - 对去除椒盐噪声特别有效
median = cv2.medianBlur(img, 5)
median2 = cv2.medianBlur(img, 9)

### 4.4 双边滤波

双边滤波同时考虑空间距离和像素值差异，在平滑噪声的同时保持边缘清晰。

参数：
- d - 邻域直径
- sigmaColor - 颜色空间标准差
- sigmaSpace - 坐标空间标准差

In [ ]:
img = cv2.imread("test.jpg")

# 双边滤波 - 保持边缘的同时平滑
bilateral = cv2.bilateralFilter(img, 9, 75, 75)

---

## 5. 形态学操作

形态学操作基于图像的形状和结构进行处理，常用于二值图像。

### 5.1 腐蚀与膨胀

- **腐蚀 (Erosion)** - 消除边界点，使白色区域缩小。可去除小的白色噪点。
- **膨胀 (Dilation)** - 增加边界点，使白色区域扩大。可填补小的黑色空洞。

### 5.2 开运算与闭运算

- **开运算 (Opening)** = 腐蚀 + 膨胀，用于去除白色噪点
- **闭运算 (Closing)** = 膨胀 + 腐蚀，用于填补黑色空洞

### 5.3 其他形态学操作

- **梯度 (Gradient)** = 膨胀 - 腐蚀，获取物体轮廓
- **顶帽 (Top Hat)** = 原图 - 开运算
- **黑帽 (Black Hat)** = 闭运算 - 原图

In [ ]:
img = cv2.imread("test.jpg", cv2.IMREAD_GRAYSCALE)

# 创建卷积核
kernel = np.ones((5, 5), np.uint8)

# 腐蚀
erosion = cv2.erode(img, kernel, iterations=1)

# 膨胀
dilation = cv2.dilate(img, kernel, iterations=1)

# 开运算（先腐蚀后膨胀）
opening = cv2.morphologyEx(img, cv2.MORPH_OPEN, kernel)

# 闭运算（先膨胀后腐蚀）
closing = cv2.morphologyEx(img, cv2.MORPH_CLOSE, kernel)

# 梯度运算
gradient = cv2.morphologyEx(img, cv2.MORPH_GRADIENT, kernel)

# 顶帽
tophat = cv2.morphologyEx(img, cv2.MORPH_TOPHAT, kernel)

# 黑帽
blackhat = cv2.morphologyEx(img, cv2.MORPH_BLACKHAT, kernel)

---

## 6. 边缘检测与梯度

### 6.1 Sobel 算子

Sobel 算子通过卷积计算图像的一阶导数，检测梯度变化。

参数：
- ksize - 卷积核大小（1, 3, 5, 7）
- dx - x方向求导阶数
- dy - y方向求导阶数

In [ ]:
img = cv2.imread("test.jpg", cv2.IMREAD_GRAYSCALE)

# Sobel 在 x 方向求导
sobelx = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=5)

# Sobel 在 y 方向求导
sobely = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=5)

# 合并 Sobel x 和 y
sobelxy = cv2.bitwise_or(sobelx, sobely)

### 6.2 Laplacian 算子

Laplacian 算子计算二阶导数，对边缘更敏感。

参数：
- ksize - 卷积核大小

In [ ]:
img = cv2.imread("test.jpg", cv2.IMREAD_GRAYSCALE)

# Laplacian 边缘检测
laplacian = cv2.Laplacian(img, cv2.CV_64F, ksize=5)

### 6.3 Canny 边缘检测

Canny 算法是多阶段边缘检测算法，是最常用的边缘检测方法。

步骤：
1. 高斯模糊降噪
2. 计算梯度强度和方向
3. 非极大值抑制
4. 双阈值检测连接边缘

参数：
- threshold1 - 低阈值
- threshold2 - 高阈值

In [ ]:
img = cv2.imread("test.jpg", cv2.IMREAD_GRAYSCALE)

# Canny 边缘检测
edges = cv2.Canny(img, 50, 150)

# Canny 边缘检测（不同阈值）
edges2 = cv2.Canny(img, 100, 200)
edges3 = cv2.Canny(img, 200, 300)

# 阈值说明：
# 低阈值：低于此值的像素被判定为非边缘
# 高阈值：高于此值的像素被判定为边缘
# 介于两者之间的像素，如果与边缘像素相连则为边缘，否则被排除

---

## 7. 图像轮廓

轮廓是连接所有边界点的曲线。OpenCV 提供查找和绘制轮廓的函数。

**注意**：查找轮廓前通常需要先进行边缘检测或阈值处理，轮廓检索模式：
- `cv2.RETR_EXTERNAL` - 只检索最外层轮廓
- `cv2.RETR_LIST` - 检索所有轮廓，不建立层级
- `cv2.RETR_TREE` - 检索所有轮廓，建立完整层级

In [ ]:
img = cv2.imread("test.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 边缘检测
edges = cv2.Canny(gray, 50, 150)

# 查找轮廓
contours, hierarchy = cv2.findContours(edges, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

# 在图像上绘制轮廓
contour_img = img.copy()
cv2.drawContours(contour_img, contours, -1, (0, 255, 0), 2)

# 绘制特定轮廓
cv2.drawContours(contour_img, contours, 0, (0, 0, 255), 2)  # 绘制第0个轮廓

### 轮廓特征

OpenCV 提供多种计算轮廓特征的方法：

- 面积：`cv2.contourArea()`
- 周长：`cv2.arcLength()`
- 边界框：`cv2.boundingRect()`
- 最小外接矩形：`cv2.minAreaRect()`
- 最小外接圆：`cv2.minEnclosingCircle()`
- 椭圆拟合：`cv2.fitEllipse()`
- 直线拟合：`cv2.fitLine()`

In [ ]:
img = cv2.imread("test.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
edges = cv2.Canny(gray, 50, 150)
contours, _ = cv2.findContours(edges, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

if contours:
    cnt = contours[0]  # 以第一个轮廓为例
    
    # 面积
    area = cv2.contourArea(cnt)
    print(f"面积: {area}")
    
    # 周长（闭合轮廓=True）
    perimeter = cv2.arcLength(cnt, True)
    print(f"周长: {perimeter}")
    
    # 边界矩形
    x, y, w, h = cv2.boundingRect(cnt)
    cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)
    
    # 最小外接矩形
    rect = cv2.minAreaRect(cnt)
    box = cv2.boxPoints(rect)
    box = np.int0(box)
    cv2.drawContours(img, [box], 0, (0, 0, 255), 2)
    
    # 最小外接圆
    (x, y), radius = cv2.minEnclosingCircle(cnt)
    center = (int(x), int(y))
    radius = int(radius)
    cv2.circle(img, center, radius, (255, 0, 0), 2)

---

## 8. 直方图

直方图是图像像素值的分布统计。

### 8.1 计算与绘制直方图

`cv2.calcHist()` 用于计算直方图。

In [ ]:
img = cv2.imread("test.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 计算灰度直方图
hist = cv2.calcHist([gray], [0], None, [256], [0, 256])

# 计算彩色图像各通道直方图
b_hist = cv2.calcHist([img], [0], None, [256], [0, 256])  # Blue
g_hist = cv2.calcHist([img], [1], None, [256], [0, 256])  # Green
r_hist = cv2.calcHist([img], [2], None, [256], [0, 256])  # Red

# 绘制直方图
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.imshow(gray, cmap="gray")
plt.title("Original Gray")
plt.subplot(1, 2, 2)
plt.plot(hist)
plt.title("Grayscale Histogram")
plt.xlim([0, 256])
plt.show()

### 8.2 直方图均衡化

直方图均衡化增强图像对比度，特别适用于曝光不足或过度的图像。

In [ ]:
img = cv2.imread("test.jpg", cv2.IMREAD_GRAYSCALE)

# 直方图均衡化
equ = cv2.equalizeHist(img)

### 8.3 CLAHE (对比度受限自适应直方图均衡化)

CLAHE 在限制对比度的同时进行自适应均衡，比全局均衡化效果更好。

In [ ]:
img = cv2.imread("test.jpg", cv2.IMREAD_GRAYSCALE)

# 创建 CLAHE 对象
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

# 应用 CLAHE
clahe_img = clahe.apply(img)

---

## 9. 图像变换

### 9.1 傅里叶变换

傅里叶变换将图像从空间域转换到频域，用于分析和处理图像的频率成分。

- 低频成分对应图像的平滑区域
- 高频成分对应图像的边缘和噪声

In [ ]:
img = cv2.imread("test.jpg", cv2.IMREAD_GRAYSCALE)

# 进行 DFT（离散傅里叶变换）
dft = cv2.dft(np.float32(img), flags=cv2.DFT_COMPLEX_OUTPUT)

# 将零点移到频谱中心
dft_shift = np.fft.fftshift(dft)

# 计算幅度谱
magnitude = 20 * np.log(cv2.magnitude(dft_shift[:, :, 0], dft_shift[:, :, 1]))

# 逆变换
idfft_shift = np.fft.ifftshift(dft_shift)
img_back = cv2.idft(idfft_shift)
img_back = cv2.magnitude(img_back[:, :, 0], img_back[:, :, 1])

### 9.2 霍夫变换 - 直线检测

霍夫变换用于检测图像中的直线和圆。

`cv2.HoughLines()` 使用标准霍夫变换检测直线。

参数：
- rho - 距离分辨率（像素）
- theta - 角度分辨率（弧度）
- threshold - 累加器阈值

In [ ]:
img = cv2.imread("test.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 边缘检测
edges = cv2.Canny(gray, 50, 150, apertureSize=3)

# 霍夫直线检测
lines = cv2.HoughLines(edges, 1, np.pi/180, 200)

# 绘制检测到的直线
for line in lines:
    rho, theta = line[0]
    a = np.cos(theta)
    b = np.sin(theta)
    x0 = a * rho
    y0 = b * rho
    x1 = int(x0 + 1000 * (-b))
    y1 = int(y0 + 1000 * (a))
    x2 = int(x0 - 1000 * (-b))
    y2 = int(y0 - 1000 * (a))
    cv2.line(img, (x1, y1), (x2, y2), (0, 0, 255), 2)

### 9.3 霍夫变换 - 概率霍夫直线检测

`cv2.HoughLinesP()` 是霍夫直线检测的优化版本，速度更快。

In [ ]:
img = cv2.imread("test.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
edges = cv2.Canny(gray, 50, 150, apertureSize=3)

# 概率霍夫直线检测
lines = cv2.HoughLinesP(edges, 1, np.pi/180, 100, minLineLength=100, maxLineGap=10)

for line in lines:
    x1, y1, x2, y2 = line[0]
    cv2.line(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

### 9.4 霍夫圆检测

`cv2.HoughCircles()` 用于检测图像中的圆。

参数：
- method - 霍夫变换方法（通常用 cv2.HOUGH_GRADIENT）
- dp - 累加器分辨率与图像分辨率的比值
- minDist - 检测到的圆心之间的最小距离
- param1 - Canny边缘检测的高阈值
- param2 - 圆心检测的累加器阈值
- minRadius - 最小半径
- maxRadius - 最大半径

In [ ]:
img = cv2.imread("test.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 中值模糊降噪
gray = cv2.medianBlur(gray, 5)

# 霍夫圆检测
circles = cv2.HoughCircles(gray, cv2.HOUGH_GRADIENT, 1, 50, 
                            param1=50, param2=30, 
                            minRadius=0, maxRadius=0)

if circles is not None:
    circles = np.uint16(np.around(circles))
    for i in circles[0, :]:
        # 绘制圆
        cv2.circle(img, (i[0], i[1]), i[2], (0, 255, 0), 2)
        # 绘制圆心
        cv2.circle(img, (i[0], i[1]), 2, (0, 0, 255), 3)

---

## 10. 模板匹配

模板匹配是在大图像中查找模板图像位置的方法。

匹配方法：
- `cv2.TM_CCOEFF` - 相关系数匹配
- `cv2.TM_CCOEFF_NORMED` - 归一化相关系数匹配
- `cv2.TM_CCORR` - 相关匹配
- `cv2.TM_CCORR_NORMED` - 归一化相关匹配
- `cv2.TM_SQDIFF` - 平方差匹配
- `cv2.TM_SQDIFF_NORMED` - 归一化平方差匹配

In [ ]:
img = cv2.imread("test.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 读取模板
template = cv2.imread("template.jpg", cv2.IMREAD_GRAYSCALE)
w, h = template.shape[::-1]

# 模板匹配
result = cv2.matchTemplate(gray, template, cv2.TM_CCOEFF_NORMED)

# 查找最佳匹配位置
min_val, max_val, min_loc, max_loc = cv2.minMaxLoc(result)

# 绘制匹配结果
top_left = max_loc
bottom_right = (top_left[0] + w, top_left[1] + h)
cv2.rectangle(img, top_left, bottom_right, (0, 255, 0), 2)

# 设定阈值，查找所有匹配位置
threshold = 0.8
locations = np.where(result >= threshold)
for pt in zip(*locations[::-1]):
    cv2.rectangle(img, pt, (pt[0] + w, pt[1] + h), (0, 0, 255), 2)

---

## 11. 图像修复与去噪

### 11.1 图像去噪

OpenCV 提供了几种去噪方法：

- `cv2.fastNlMeansDenoising()` - 非局部均值去噪（效果好但慢）
- `cv2.fastNlMeansDenoisingColored()` - 彩色图像去噪
- `cv2.fastNlMeansDenoisingMulti()` - 视频序列去噪

参数：
- h - 滤镜强度
- templateWindowSize - 模板窗口大小（默认7）
- searchWindowSize - 搜索窗口大小（默认21）

In [ ]:
img = cv2.imread("test.jpg")

# 非局部均值去噪 - 彩色图像
denoised = cv2.fastNlMeansDenoisingColored(img, None, 10, 10, 7, 21)

# 灰度图像去噪
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
denoised_gray = cv2.fastNlMeansDenoising(gray, None, 10, 7, 21)

### 11.2 图像修复

`cv2.inpaint()` 可以修复图像中的损坏区域或去除不需要的物体。

方法：
- `cv2.INPAINT_TELEA` - 基于电信的修复算法
- `cv2.INPAINT_NS` - 基于Navier-Stokes的修复算法

In [ ]:
img = cv2.imread("test.jpg")
mask = cv2.imread("mask.jpg", cv2.IMREAD_GRAYSCALE)  # 损坏区域掩码

# 图像修复
result = cv2.inpaint(img, mask, 3, cv2.INPAINT_TELEA)

---

## 12. 几何变换

### 12.1 裁剪

直接使用 NumPy 数组切片进行图像裁剪。

In [ ]:
img = cv2.imread("test.jpg")

# 裁剪图像 [y1:y2, x1:x2]
cropped = img[100:400, 200:500]

# 裁剪中心区域
h, w = img.shape[:2]
center_h, center_w = h // 2, w // 2
cropped_center = img[center_h-100:center_h+100, center_w-100:center_w+100]

### 12.2 翻转

`cv2.flip()` 可以水平、垂直或两个方向同时翻转图像。

In [ ]:
img = cv2.imread("test.jpg")

# 水平翻转 (flipCode = 1)
flipped_horizontal = cv2.flip(img, 1)

# 垂直翻转 (flipCode = 0)
flipped_vertical = cv2.flip(img, 0)

# 同时水平和垂直翻转 (flipCode = -1)
flipped_both = cv2.flip(img, -1)

### 12.3 旋转

使用 `cv2.rotate()` 可以快速旋转图像90度的倍数。

- `cv2.ROTATE_90_CLOCKWISE` - 顺时针90度
- `cv2.ROTATE_90_COUNTERCLOCKWISE` - 逆时针90度
- `cv2.ROTATE_180` - 180度

In [ ]:
img = cv2.imread("test.jpg")

# 顺时针旋转90度
rotated_cw = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)

# 逆时针旋转90度
rotated_ccw = cv2.rotate(img, cv2.ROTATE_90_COUNTERCLOCKWISE)

# 旋转180度
rotated_180 = cv2.rotate(img, cv2.ROTATE_180)

---

## 13. 颜色检测与追踪

在 HSV 颜色空间中进行颜色检测比在 BGR 空间更准确。

步骤：
1. 将图像从 BGR 转换到 HSV
2. 定义HSV颜色的阈值范围
3. 创建掩码提取目标颜色区域

In [ ]:
img = cv2.imread("test.jpg")
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# 定义蓝色的HSV阈值范围
lower_blue = np.array([100, 50, 50])
upper_blue = np.array([130, 255, 255])

# 创建掩码
mask = cv2.inRange(hsv, lower_blue, upper_blue)

# 应用掩码
result = cv2.bitwise_and(img, img, mask=mask)

### 查找HSV颜色范围

可以使用以下代码实时查找特定颜色的HSV范围。

In [ ]:
# 查找特定颜色的HSV范围
# green = np.uint8([[[B, G, R]]])
# hsv_green = cv2.cvtColor(green, cv2.COLOR_BGR2HSV)

# 常用颜色的HSV阈值范围（供参考）
# 白色: H:0-180, S:0-30, V:200-255
# 黑色: H:0-180, S:0-255, V:0-30
# 红色: H:0-10 or 170-180, S:100-255, V:100-255
# 绿色: H:35-85, S:50-255, V:50-255
# 蓝色: H:90-130, S:50-255, V:50-255
# 黄色: H:15-35, S:50-255, V:100-255

---

## 14. 视频处理基础

### 14.1 读取视频

使用 `cv2.VideoCapture()` 读取视频文件或摄像头。

常用方法：
- `read()` - 读取下一帧
- `isOpened()` - 检查是否打开
- `release()` - 释放资源

In [ ]:
# 读取视频文件
cap = cv2.VideoCapture("video.mp4")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    # 处理每一帧
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    cv2.imshow("frame", gray)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

### 14.2 保存视频

使用 `cv2.VideoWriter()` 保存视频。

In [ ]:
cap = cv2.VideoCapture(0)  # 打开摄像头

# 定义编解码器
fourcc = cv2.VideoWriter_fourcc(*"XVID")
out = cv2.VideoWriter("output.avi", fourcc, 20.0, (640, 480))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    out.write(frame)  # 写入帧
    cv2.imshow("frame", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
out.release()
cv2.destroyAllWindows()

---

## 15. 绘图功能

OpenCV 提供多种绘图函数：

- `cv2.line()` - 画直线
- `cv2.rectangle()` - 画矩形
- `cv2.circle()` - 画圆
- `cv2.ellipse()` - 画椭圆
- `cv2.putText()` - 绘制文本

In [ ]:
img = np.zeros((512, 512, 3), np.uint8)

# 画线 (图像, 起点, 终点, 颜色, 粗细)
cv2.line(img, (0, 0), (511, 511), (255, 0, 0), 5)

# 画矩形 (图像, 左上角, 右下角, 颜色, 粗细)
cv2.rectangle(img, (384, 0), (510, 128), (0, 255, 0), 3)

# 画圆 (图像, 圆心, 半径, 颜色, 粗细)
cv2.circle(img, (447, 63), 63, (0, 0, 255), -1)  # -1表示填充

# 画椭圆 (图像, 圆心, (长轴, 短轴), 旋转角度, 起始角度, 结束角度, 颜色, 粗细)
cv2.ellipse(img, (256, 256), (100, 50), 0, 0, 180, (255, 255, 0), 3)

# 绘制文本
font = cv2.FONT_HERSHEY_SIMPLEX
cv2.putText(img, "OpenCV", (10, 500), font, 2, (255, 255, 255), 2, cv2.LINE_AA)

---

## 16. 图像拼接与混合

### 16.1 图像加法

使用 `cv2.add()` 或直接用 `+` 运算符可以合并图像。

注意：
- `cv2.add()` 会进行饱和运算（超过255的值会截断为255）
- `+` 运算符会进行模运算（256会变成0）

In [ ]:
img1 = cv2.imread("image1.jpg")
img2 = cv2.imread("image2.jpg")

# OpenCV 加法（饱和运算）
added_cv = cv2.add(img1, img2)

# NumPy 加法（模运算）
added_np = img1 + img2

### 16.2 图像混合

`cv2.addWeighted()` 可以在两幅图像之间进行加权混合：

公式：`dst = alpha * img1 + beta * img2 + gamma`

其中 alpha + beta 应该等于1（或gamma作为补偿）。

In [ ]:
img1 = cv2.imread("image1.jpg")
img2 = cv2.imread("image2.jpg")

# 确保图像尺寸相同
img2 = cv2.resize(img2, (img1.shape[1], img1.shape[0]))

# 加权混合
blended = cv2.addWeighted(img1, 0.7, img2, 0.3, 0)

### 16.3 图像拼接

使用 `cv2.hconcat()` 水平拼接，`cv2.vconcat()` 垂直拼接。

In [ ]:
img1 = cv2.imread("image1.jpg")
img2 = cv2.imread("image2.jpg")

# 水平拼接
horizontal = cv2.hconcat([img1, img2])

# 垂直拼接
vertical = cv2.vconcat([img1, img2])

---

## 17. 图像掩膜操作

掩膜（Mask）是一个与图像尺寸相同的二值图像，用于限定图像的处理区域。

常用操作：
- `cv2.bitwise_and()` - 与操作
- `cv2.bitwise_or()` - 或操作
- `cv2.bitwise_xor()` - 异或操作
- `cv2.bitwise_not()` - 非操作

In [ ]:
img = cv2.imread("test.jpg")

# 创建掩膜（圆形区域）
mask = np.zeros(img.shape[:2], np.uint8)
center = (img.shape[1]//2, img.shape[0]//2)
radius = 100
cv2.circle(mask, center, radius, 255, -1)

# 使用掩膜提取区域
masked = cv2.bitwise_and(img, img, mask=mask)

# 创建带透明度的掩膜
mask_float = np.float32(mask) / 255.0
result = img * mask_float[:, :, np.newaxis]

---

## 18. 常用功能汇总表

| 功能类别 | 函数 | 说明 |
|---------|------|------|
| **图像读写** | `cv2.imread()` | 读取图像 |
| | `cv2.imwrite()` | 保存图像 |
| **颜色转换** | `cv2.cvtColor()` | 颜色空间转换 |
| | `cv2.split()` | 分离通道 |
| | `cv2.merge()` | 合并通道 |
| **几何变换** | `cv2.resize()` | 缩放图像 |
| | `cv2.warpAffine()` | 仿射变换 |
| | `cv2.warpPerspective()` | 透视变换 |
| | `cv2.getRotationMatrix2D()` | 获取旋转矩阵 |
| | `cv2.flip()` | 翻转图像 |
| **图像滤波** | `cv2.blur()` | 平均模糊 |
| | `cv2.GaussianBlur()` | 高斯模糊 |
| | `cv2.medianBlur()` | 中值模糊 |
| | `cv2.bilateralFilter()` | 双边滤波 |
| **形态学** | `cv2.erode()` | 腐蚀 |
| | `cv2.dilate()` | 膨胀 |
| | `cv2.morphologyEx()` | 组合形态学操作 |
| **边缘检测** | `cv2.Sobel()` | Sobel算子 |
| | `cv2.Laplacian()` | Laplacian算子 |
| | `cv2.Canny()` | Canny边缘检测 |
| **轮廓** | `cv2.findContours()` | 查找轮廓 |
| | `cv2.drawContours()` | 绘制轮廓 |
| **直方图** | `cv2.calcHist()` | 计算直方图 |
| | `cv2.equalizeHist()` | 直方图均衡化 |
| | `cv2.createCLAHE()` | 自适应直方图均衡化 |
| **霍夫变换** | `cv2.HoughLines()` | 霍夫直线检测 |
| | `cv2.HoughLinesP()` | 概率霍夫直线检测 |
| | `cv2.HoughCircles()` | 霍夫圆检测 |
| **模板匹配** | `cv2.matchTemplate()` | 模板匹配 |
| **图像修复** | `cv2.inpaint()` | 图像修复 |
| **绘图** | `cv2.line()` | 画线 |
| | `cv2.rectangle()` | 画矩形 |
| | `cv2.circle()` | 画圆 |
| | `cv2.putText()` | 绘制文本 |
| **位运算** | `cv2.bitwise_and()` | 按位与 |
| | `cv2.bitwise_or()` | 按位或 |
| | `cv2.bitwise_xor()` | 按位异或 |
| | `cv2.bitwise_not()` | 按位非 |
| **阈值处理** | `cv2.threshold()` | 固定阈值 |
| | `cv2.adaptiveThreshold()` | 自适应阈值 |

---

## 19. 学习建议

### 入门路径

1. **基础操作**：图像读写、显示、保存
2. **图像变换**：缩放、旋转、翻转、裁剪
3. **颜色空间**：BGR、RGB、灰度、HSV 转换
4. **图像滤波**：模糊、锐化、边缘检测
5. **形态学操作**：腐蚀、膨胀、开闭运算
6. **轮廓处理**：查找、绘制、特征提取
7. **实战应用**：车牌识别、人脸检测、物体追踪

### 进阶方向

- 机器学习集成（scikit-learn, TensorFlow, PyTorch）
- 深度学习（使用预训练模型进行图像分类、目标检测）
- 视频分析（光流、背景建模、动作识别）
- 3D重建（立体视觉、深度估计）
- 性能优化（GPU加速、多线程）